# 👥 HR Analytics — Employee Attrition Dashboard
### End-to-End HR Analysis | Python · SQL · Tableau
**Author:** Rahul Sharma | Datamites — Data Analytics Certification
**Dataset:** 1,470 employees | 28 features | IBM HR Analytics style


## 1. Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 120})
PALETTE = {'Yes': '#C73E1D', 'No': '#2E86AB'}

df = pd.read_csv('../data/hr_attrition.csv')
df['Attrition_Flag'] = (df['Attrition'] == 'Yes').astype(int)
attr_rate = df['Attrition_Flag'].mean() * 100

print(f"Shape: {df.shape}")
print(f"Attrition Rate: {attr_rate:.1f}%")
print(f"Employees Left: {df['Attrition_Flag'].sum()}")
df.head()

## 2. Data Overview & Quality Check

In [ ]:
print("Data Types:")
print(df.dtypes)
print(f"\nNull Values: {df.isnull().sum().sum()}")
print(f"\nDepartment Distribution:")
print(df['Department'].value_counts())
df.describe()

## 3. Overall Attrition Distribution

In [ ]:
counts = df['Attrition'].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(11,5))
fig.suptitle('Overall Attrition Distribution', fontsize=14, fontweight='bold', color='#1F4E79')

axes[0].bar(['Stayed','Left'], [counts['No'], counts['Yes']],
            color=['#2E86AB','#C73E1D'], width=0.4, edgecolor='white')
axes[0].set_title('Employee Count')
for i, (v, l) in enumerate(zip([counts['No'],counts['Yes']],[counts['No'],counts['Yes']])):
    axes[0].text(i, v+10, f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center', fontsize=11)

axes[1].pie([counts['No'],counts['Yes']], labels=['Stayed','Left'],
            autopct='%1.1f%%', colors=['#2E86AB','#C73E1D'],
            startangle=90, wedgeprops={'edgecolor':'white','linewidth':2})
axes[1].set_title('Attrition Rate')
plt.tight_layout(); plt.show()

## 4. Attrition by Department

In [ ]:
dept = df.groupby('Department').agg(
    total=('Attrition_Flag','count'), left=('Attrition_Flag','sum')).reset_index()
dept['rate'] = dept['left']/dept['total']*100
dept = dept.sort_values('rate', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13,5))
axes[0].bar(dept['Department'], dept['rate'],
            color=['#C73E1D','#F18F01','#2E86AB'], width=0.5, edgecolor='white')
axes[0].set_title('Attrition Rate by Department')
axes[0].axhline(attr_rate, color='gray', linestyle='--', linewidth=1.5, label=f'Avg ({attr_rate:.1f}%)')
axes[0].legend()
for i, v in enumerate(dept['rate']):
    axes[0].text(i, v+0.5, f'{v:.1f}%', ha='center', fontsize=11, fontweight='bold')

axes[1].bar(dept['Department'], dept['total'], label='Total', color='#D6E4F0', edgecolor='#1F4E79')
axes[1].bar(dept['Department'], dept['left'], label='Left', color='#C73E1D', alpha=0.85)
axes[1].set_title('Headcount vs Attrition'); axes[1].legend()
plt.tight_layout(); plt.show()

## 5. Satisfaction Scores Analysis

In [ ]:
sat_cols = ['JobSatisfaction','EnvironmentSatisfaction','WorkLifeBalance','JobInvolvement']
sat_labels = ['Job Satisfaction','Env. Satisfaction','Work-Life Balance','Job Involvement']

fig, axes = plt.subplots(2, 2, figsize=(14,10))
axes = axes.flatten()
for ax, col, lbl in zip(axes, sat_cols, sat_labels):
    grp = (df.groupby(col)['Attrition_Flag'].mean()*100).reset_index()
    grp.columns = [col,'rate']
    bar_colors = ['#C73E1D' if v > attr_rate else '#2E86AB' for v in grp['rate']]
    ax.bar(grp[col].astype(str), grp['rate'], color=bar_colors, width=0.5, edgecolor='white')
    ax.set_title(f'{lbl} vs Attrition Rate')
    ax.set_xlabel('Score (1=Low, 4=High)'); ax.set_ylabel('Attrition Rate (%)')
    ax.axhline(attr_rate, color='gray', linestyle='--', linewidth=1.2)
    for i, v in enumerate(grp['rate']):
        ax.text(i, v+0.5, f'{v:.1f}%', ha='center', fontsize=10)
plt.tight_layout(); plt.show()

## 6. Overtime & Income Impact

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13,5))

# Overtime
ot = (df.groupby('OverTime')['Attrition_Flag'].mean()*100).reset_index()
ot.columns = ['OverTime','rate']
axes[0].bar(ot['OverTime'], ot['rate'], color=['#C73E1D','#2E86AB'], width=0.4, edgecolor='white')
axes[0].set_title('Attrition Rate by Overtime'); axes[0].set_ylabel('Attrition Rate (%)')
axes[0].axhline(attr_rate, color='gray', linestyle='--')
for i, v in enumerate(ot['rate']):
    axes[0].text(i, v+0.5, f'{v:.1f}%', ha='center', fontsize=12, fontweight='bold')

# Income
sns.boxplot(data=df, x='Attrition', y='MonthlyIncome', palette=PALETTE, ax=axes[1], width=0.4)
axes[1].set_title('Monthly Income by Attrition')
axes[1].set_xlabel('Attrition'); axes[1].set_ylabel('Monthly Income')
plt.tight_layout(); plt.show()

print(f"Attrition with Overtime:    {df[df.OverTime=='Yes']['Attrition_Flag'].mean()*100:.1f}%")
print(f"Attrition without Overtime: {df[df.OverTime=='No']['Attrition_Flag'].mean()*100:.1f}%")
print(f"Avg Income (Left):   ₹{df[df.Attrition=='Yes']['MonthlyIncome'].mean():,.0f}")
print(f"Avg Income (Stayed): ₹{df[df.Attrition=='No']['MonthlyIncome'].mean():,.0f}")

## 7. Correlation Heatmap

In [ ]:
num_cols = ['Age','MonthlyIncome','TotalWorkingYears','YearsAtCompany',
            'JobSatisfaction','WorkLifeBalance','EnvironmentSatisfaction',
            'DistanceFromHome','NumCompaniesWorked','Attrition_Flag']
corr = df[num_cols].corr()
fig, ax = plt.subplots(figsize=(11,8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            mask=np.triu(np.ones_like(corr, dtype=bool)), ax=ax, linewidths=0.5, square=True)
ax.set_title('Correlation Heatmap', fontsize=13, fontweight='bold', color='#1F4E79')
plt.tight_layout(); plt.show()

## 8. Key Business Insights & Recommendations

| # | Insight | Recommendation |
|---|---------|----------------|
| 1 | **Sales dept has 34.7% attrition** — highest | Review Sales compensation & incentive structure |
| 2 | **Overtime employees churn 2x more** | Cap mandatory overtime; hire additional staff |
| 3 | **Low job satisfaction = highest churn** | Quarterly engagement surveys + action plans |
| 4 | **First 3 years = highest attrition risk** | Structured mentorship for new joiners |
| 5 | **Single employees churn more** | Work-life balance & social engagement programs |
| 6 | **Frequent travelers churn more** | Limit travel frequency; offer travel allowances |
| 7 | **Low income = high attrition** | Benchmark salaries against market; revise bands |
